# Beag NIST Compliance Model Training

Train a domain-specific LoRA adapter that maps documents to NIST 800-53 / CSF / CMMC controls.

**Hardware**: Kaggle T4 GPU (16 GB VRAM)
**Base model**: Qwen2.5-7B-Instruct (4-bit QLoRA)
**Training recipe**: CISPO loss + interleaved batching
**Runtime**: ~15-25 min

## 1. Clone repo

In [ ]:

import os, sys, json, time, subprocess
# Prevent CUDA OOM fragmentation (critical for T4)
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
os.environ["OMP_NUM_THREADS"] = "4"
from pathlib import Path

REPO_URL = "https://github.com/YOUR_ORG/beag-training.git"  # <-- replace with actual
REPO_DIR = Path("/kaggle/working/beag-training")

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    print("Repo already cloned.")

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
print(f"Working dir: {os.getcwd()}")

## 2. Set API key (MUST run first — before any project imports)

In [ ]:
# Option A: Kaggle Secrets (Add-on -> Secrets -> DEEPSEEK_API_KEY)
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["DEEPSEEK_API_KEY"] = UserSecretsClient().get_secret("DEEPSEEK_API_KEY")
    print("API key loaded from Kaggle Secrets.")
except Exception:
    pass

# Option B: Paste key directly (uncomment below)
# os.environ["DEEPSEEK_API_KEY"] = "sk-..."

api_key = os.environ.get("DEEPSEEK_API_KEY", "")
if not api_key:
    raise RuntimeError("DEEPSEEK_API_KEY not set. Use Kaggle Secrets or paste it above.")
print(f"API key: {'***' + api_key[-4:] if len(api_key) > 4 else 'SET'}")

## 3. Install dependencies

In [ ]:
# GPU check
try:
    subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], check=True)
except Exception:
    print("WARNING: No GPU detected.")

In [ ]:
!pip install -q --upgrade pip setuptools wheel
!pip install -q unsloth
!pip install -q openai pydantic-settings python-dotenv pyyaml scikit-learn pyarrow sentence-transformers

## 4. Import project modules

In [ ]:
from data import DataGenerator, GeneratorConfig
from data.validator import filter_valid
from data.augment import DataAugmenter
from frameworks.catalog import Framework, load_catalog
from model.output_schema import TrainingExample
from frontier.deepseek import DeepSeekClient

print("Project imports OK")

# Quick API connectivity test (uses os.environ directly, not settings cache)
from openai import AsyncOpenAI
test_client = AsyncOpenAI(
    base_url="https://api.deepseek.com",
    api_key=api_key,
    timeout=15,
)
test_resp = await test_client.chat.completions.create(
    model="deepseek-chat",
    messages=[{"role": "user", "content": "say ok"}],
    max_tokens=10,
)
print(f"DeepSeek API: {test_resp.choices[0].message.content}")

## 5. Generate synthetic training data

In [ ]:
NUM_EXAMPLES = 200
CONCURRENCY = 10
AMBIGUITY_PROB = 0.5

output_dir = REPO_DIR / "output"
output_dir.mkdir(exist_ok=True)

cat = load_catalog()
print(f"Catalog: {len(list(cat.list_by_framework(Framework.NIST_800_53)))} NIST 800-53 controls")

In [ ]:
config = GeneratorConfig(
    total_examples=NUM_EXAMPLES,
    frameworks=list(Framework),
    concurrency=CONCURRENCY,
    max_mappings_per_example=6,
    min_mappings_per_example=3,
    ambiguity_prob=AMBIGUITY_PROB,
)

# Pass API key directly instead of relying on settings singleton
gen = DataGenerator(
    client=DeepSeekClient(api_key=api_key),
    catalog=cat,
    config=config,
)

print(f"Generating {NUM_EXAMPLES} examples...")
t0 = time.monotonic()
examples = await gen.generate_all_async()
examples, invalid = filter_valid(examples, cat)
print(f"Valid: {len(examples)}  Invalid: {len(invalid)}  Time: {time.monotonic()-t0:.0f}s")

In [ ]:
if not examples:
    raise RuntimeError("0 valid examples generated. Check API key connectivity.")

# Augment ~20%
augmenter = DataAugmenter(cat)
augmented = []
for ex in examples[:max(1, len(examples)//5)]:
    augmented.extend(augmenter.augment(ex))
aug_valid, _ = filter_valid(augmented, cat)

all_examples = examples + aug_valid

train_path = output_dir / "generated_augmented.jsonl"
with open(train_path, "w") as f:
    for ex in all_examples:
        record = {
            "text": json.dumps(ex.input_json),
            "mappings": [m.to_dict() for m in ex.mappings],
            "text_context": ex.text_context,
        }
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(all_examples)} examples to {train_path}")

## 6. Train with QLoRA + CISPO

In [ ]:
import torch
from onprem.unsloth_adapter import UnslothAdapter
from onprem.train import OnPremTrainer, load_dataset_from_jsonl

MODEL_TIER = "starter"  # Qwen2.5-3B-Instruct (~6 GB VRAM FP16, fits T4)
LORA_RANK = 32
BATCH_SIZE = 1
EPOCHS = 3
MAX_SEQ_LENGTH = 2048

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM free: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

In [ ]:
adapter = UnslothAdapter(
    model_tier=MODEL_TIER,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=False,  # 4-bit broken on CUDA 12.8; FP16 fits T4
    lora_rank=LORA_RANK,
    lora_alpha=16,
)
model, tokenizer = adapter.load()
print(f"Loaded: {adapter.model_id}")
print(f"VRAM used: {torch.cuda.memory_allocated()/1e9:.1f} GB")

In [ ]:
records = load_dataset_from_jsonl(str(train_path))
print(f"Training on {len(records)} records")

trainer_config = {
    "batch_size": BATCH_SIZE,
    "num_epochs": EPOCHS,
    "max_seq_length": MAX_SEQ_LENGTH,
    "learning_rate": 2e-5,
        "gradient_accumulation": 8,  # compensate for batch_size=1
    "warmup_steps": 50,
    "max_steps": 500,
    "output_dir": str(output_dir / "checkpoints"),
    "task_type": "nist_multi_label",
    "catalog": cat,
}

In [ ]:
trainer = OnPremTrainer(model, tokenizer, config=trainer_config)
t0 = time.monotonic()
result = trainer.train(records)
elapsed = time.monotonic() - t0

print(f"\nTraining done ({elapsed:.0f}s)")
print(f"  Steps: {result['total_steps']}")
print(f"  Best loss: {result['best_loss']:.4f}")
print(f"  Final loss: {result['final_loss']:.4f}")

In [ ]:
import matplotlib.pyplot as plt
history = result.get("history", {})
if history.get("loss"):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
    ax1.plot(history["loss"])
    ax1.set_title("Training Loss")
    ax2.plot(history["lr"])
    ax2.set_title("Learning Rate (cosine warmup)")
    plt.tight_layout()
    plt.savefig(str(output_dir / "training_curves.png"), dpi=100)
    plt.show()

## 7. Save model

In [ ]:
adapter_dir = Path("/kaggle/working/beag-nist-adapter")
adapter.save_adapter(str(adapter_dir))

summary = {
    "base_model": adapter.model_id,
    "tier": MODEL_TIER,
    "lora_rank": LORA_RANK,
    "examples": len(records),
    "epochs": EPOCHS,
    "best_loss": result["best_loss"],
    "final_loss": result["final_loss"],
    "total_steps": result["total_steps"],
}
with open(adapter_dir / "training_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

size_mb = sum(f.stat().st_size for f in adapter_dir.rglob('*') if f.is_file()) / 1e6
print(f"Saved to: {adapter_dir}")
print(f"Size: {size_mb:.1f} MB")

## 8. Quick inference test

In [ ]:
from unsloth import FastLanguageModel

model_inf, tok_inf = FastLanguageModel.from_pretrained(
    model_name=str(adapter_dir),
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=False,  # 4-bit broken on CUDA 12.8; FP16 fits T4
)
FastLanguageModel.for_inference(model_inf)

test_doc = json.dumps({
    "type": "policy",
    "title": "Access Control Policy",
    "text": "All users must authenticate via multi-factor authentication before accessing protected systems.",
})

messages = [
    {"role": "system", "content": "You map policy documents to NIST 800-53 controls. Return a JSON array of control mappings with control_id, framework, confidence, and reasoning."},
    {"role": "user", "content": f"Map this document to relevant controls:\n\n{test_doc}"},
]

inputs = tok_inf.apply_chat_template(messages, tokenize=True, return_tensors="pt").to("cuda")
outputs = model_inf.generate(inputs, max_new_tokens=512, temperature=0.1)
response = tok_inf.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)

print("Model prediction:")
try:
    parsed = json.loads(response)
    print(json.dumps(parsed, indent=2))
except json.JSONDecodeError:
    print(response)